# XGBoost vs Linear Regression: Used Car Price Prediction

Predicting `selling_price` on the CarDekho used car dataset (4,340 listings), comparing XGBoost against Linear Regression across two rounds: before and after adding a `brand` feature. See the README for the full narrative.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

df = pd.read_csv('CAR DETAILS FROM CAR DEKHO.csv')
display(df.head(5))

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


## Round 1 & 2: baseline models, no `brand` feature

`owner` is label-encoded (it has a genuine order). `fuel`, `seller_type`, `transmission` are one-hot encoded with `drop_first=True` to avoid the dummy variable trap. `name` is dropped here since `brand` hasn't been extracted yet.

In [2]:
owner_map = {
    'Test Drive Car': 0, 'First Owner': 1, 'Second Owner': 2,
    'Third Owner': 3, 'Fourth & Above Owner': 4,
}
df['owner'] = df['owner'].map(owner_map)

df_no_brand = df.drop('name', axis=1)
df_encoded_a = pd.get_dummies(df_no_brand, columns=['fuel', 'seller_type', 'transmission'], drop_first=True)

y_a = df_encoded_a['selling_price']
X_a = df_encoded_a.drop('selling_price', axis=1)
Xa_train, Xa_test, ya_train, ya_test = train_test_split(X_a, y_a, test_size=0.2, random_state=5)

scaler_a = StandardScaler()
Xa_train_scaled = scaler_a.fit_transform(Xa_train)
Xa_test_scaled = scaler_a.transform(Xa_test)

In [3]:
lr_a = LinearRegression()
lr_a.fit(Xa_train_scaled, ya_train)
pred_lr_a = lr_a.predict(Xa_test_scaled)

xgb_a_default = xgb.XGBRegressor(random_state=5)
xgb_a_default.fit(Xa_train, ya_train)
pred_xgb_a_default = xgb_a_default.predict(Xa_test)

print("linreg rmse:", root_mean_squared_error(ya_test, pred_lr_a))
print("xgb (default) rmse:", root_mean_squared_error(ya_test, pred_xgb_a_default))

linreg rmse: 468111.7138285277
xgb (default) rmse: 444439.28125


### Tuning, with early stopping

Rather than hand-picking `n_estimators`, a validation split lets early stopping find the right number of trees automatically.

In [4]:
Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(Xa_train, ya_train, test_size=0.15, random_state=5)

xgb_a_tuned = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.01, max_depth=5,
    subsample=0.8, colsample_bytree=0.7, random_state=5,
    early_stopping_rounds=30, eval_metric='rmse'
)
xgb_a_tuned.fit(Xa_tr, ya_tr, eval_set=[(Xa_val, ya_val)], verbose=False)

print("best iteration:", xgb_a_tuned.best_iteration)

pred_xgb_a_tuned = xgb_a_tuned.predict(Xa_test)
pred_xgb_a_tuned_train = xgb_a_tuned.predict(Xa_tr)

print("xgb (tuned) train rmse:", root_mean_squared_error(ya_tr, pred_xgb_a_tuned_train))
print("xgb (tuned) test rmse:", root_mean_squared_error(ya_test, pred_xgb_a_tuned))

best iteration: 263
xgb (tuned) train rmse: 278582.65625
xgb (tuned) test rmse: 431672.46875


**Bug note:** an earlier version of this notebook computed \`RMSE(tuned_pred, lin_pred)\` here instead of \`RMSE(tuned_pred, y_test)\`, comparing two prediction arrays against each other instead of against the true values. That produced a misleadingly good number. Always compare predictions to the true target, never to another model's predictions.

## Round 3: adding `brand`, and hitting a tuning ceiling

`brand` is extracted from the free-text `name` column. Rare brands (under 15 listings) are bucketed into `Other` so the model isn't asked to learn a coefficient or split from a handful of rows.

In [5]:
df['brand'] = df['name'].str.split().str[0]
brand_counts = df['brand'].value_counts()
rare_brands = brand_counts[brand_counts < 15].index
df['brand'] = df['brand'].replace(rare_brands, 'Other')
print(df['brand'].value_counts())

brand
Maruti           1280
Hyundai           821
Mahindra          365
Tata              361
Honda             252
Ford              238
Toyota            206
Chevrolet         188
Renault           146
Volkswagen        107
Skoda              68
Nissan             64
Audi               60
BMW                39
Datsun             37
Fiat               37
Other              36
Mercedes-Benz      35
Name: count, dtype: int64


In [6]:
df_encoded_b = pd.get_dummies(
    df.drop('name', axis=1),
    columns=['fuel', 'seller_type', 'transmission', 'brand'],
    drop_first=True
)

y_b = df_encoded_b['selling_price']
X_b = df_encoded_b.drop('selling_price', axis=1)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_b, y_b, test_size=0.2, random_state=5)

scaler_b = StandardScaler()
Xb_train_scaled = scaler_b.fit_transform(Xb_train)
Xb_test_scaled = scaler_b.transform(Xb_test)

lr_b = LinearRegression()
lr_b.fit(Xb_train_scaled, yb_train)
pred_lr_b = lr_b.predict(Xb_test_scaled)

xgb_b_default = xgb.XGBRegressor(random_state=5)
xgb_b_default.fit(Xb_train, yb_train)
pred_xgb_b_default = xgb_b_default.predict(Xb_test)

print("linreg (+brand) rmse:", root_mean_squared_error(yb_test, pred_lr_b))
print("xgb default (+brand) rmse:", root_mean_squared_error(yb_test, pred_xgb_b_default))

linreg (+brand) rmse: 402712.1884252318
xgb default (+brand) rmse: 362179.53125


Does tuning help further, now that `brand` supplies real signal? A full grid search (5-fold CV) checks this properly rather than hand-picking a config that happens to look good.

In [7]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 6],
    'learning_rate': [0.05, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.7, 1.0],
}
grid_search = GridSearchCV(
    xgb.XGBRegressor(random_state=5), param_grid,
    scoring='neg_root_mean_squared_error', cv=5, n_jobs=-1
)
grid_search.fit(Xb_train, yb_train)

xgb_b_tuned = grid_search.best_estimator_
pred_xgb_b_tuned = xgb_b_tuned.predict(Xb_test)

print("best params:", grid_search.best_params_)
print("grid-search-tuned (+brand) test rmse:", root_mean_squared_error(yb_test, pred_xgb_b_tuned))
print("default (+brand) test rmse for comparison:", root_mean_squared_error(yb_test, pred_xgb_b_default))

best params: {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 300, 'subsample': 1.0}
grid-search-tuned (+brand) test rmse: 368312.40625
default (+brand) test rmse for comparison: 362179.53125


The tuned and default models land within noise of each other, unlike Round 2 where tuning gave a real gain. Once `brand` supplied the missing signal, this dataset's learnable ceiling was basically reached. **Default** XGBoost is used as the final model going forward, since it's simplest and ties the tuned result.

## Results summary

In [8]:
results = pd.DataFrame({
    'Model': [
        'Linear Regression (no brand)', 'XGBoost default (no brand)', 'XGBoost tuned (no brand)',
        'Linear Regression (+brand)', 'XGBoost default (+brand)', 'XGBoost grid-tuned (+brand)',
    ],
    'Test RMSE': [
        root_mean_squared_error(ya_test, pred_lr_a),
        root_mean_squared_error(ya_test, pred_xgb_a_default),
        root_mean_squared_error(ya_test, pred_xgb_a_tuned),
        root_mean_squared_error(yb_test, pred_lr_b),
        root_mean_squared_error(yb_test, pred_xgb_b_default),
        root_mean_squared_error(yb_test, pred_xgb_b_tuned),
    ]
})
display(results)

,Model,Test RMSE
0,Linear Regression (no brand),468111.713829
1,XGBoost default (no brand),444439.281250
2,XGBoost tuned (no brand),431672.468750
3,Linear Regression (+brand),402712.188425
4,XGBoost default (+brand),362179.531250
5,XGBoost grid-tuned (+brand),368312.406250


## Why does XGBoost win? SHAP interaction values

Trees can learn `brand x year` interactions automatically (different brands depreciate on different curves), without any manual interaction terms. SHAP interaction values, computed on the final default (+brand) model, confirm the model is actually exploiting this.

In [9]:
explainer = shap.TreeExplainer(xgb_b_default)
shap_interaction = explainer.shap_interaction_values(Xb_test)

year_idx = list(Xb_test.columns).index('year')
brand_cols = [col for col in Xb_test.columns if col.startswith('brand_')]

interaction_strengths = {}
for col in brand_cols:
    col_idx = list(Xb_test.columns).index(col)
    interaction_strengths[col] = abs(shap_interaction[:, year_idx, col_idx]).mean()

pd.Series(interaction_strengths).sort_values(ascending=False)

brand_Maruti           6474.807129
brand_Mercedes-Benz    5164.455078
brand_Toyota           4767.559570
brand_Tata             4083.482666
brand_BMW              3873.874023
brand_Other            2514.721680
brand_Mahindra         2348.644531
brand_Hyundai          2259.952881
brand_Chevrolet        1982.879761
brand_Ford             1736.810181
brand_Renault          1736.110596
brand_Volkswagen       1386.246216
brand_Honda             915.345032
brand_Datsun            704.313843
brand_Skoda             619.981995
brand_Nissan            399.260590
brand_Fiat              314.402863
dtype: float32